In [1]:
import os
import sys
from pyspark.sql import SparkSession, types

In [2]:
# 1. 환경 설정 (사용자님 시스템 경로 확인)
os.environ["JAVA_HOME"] = "C:/tools/jdk17"
os.environ["HADOOP_HOME"] = "C:/tools/hadoop-3.2.0"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin;" + os.environ["HADOOP_HOME"] + "/bin;" + os.environ["PATH"]
os.environ['PYSPARK_PYTHON'] = sys.executable

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('taxi_schema_final') \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .getOrCreate()

In [4]:
# 2. 스키마 정의 (강의 내용 + 실습용 PULocationID 변조 포함)
green_schema = types.StructType([
    types.StructField("VendorID", types.LongType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.DoubleType(), True),
    types.StructField("PULocationID", types.LongType(), True),
    types.StructField("DOLocationID", types.LongType(), True),
    types.StructField("passenger_count", types.DoubleType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.DoubleType(), True),
    types.StructField("trip_type", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

yellow_schema = types.StructType([
    types.StructField("VendorID", types.LongType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.DoubleType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.DoubleType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.LongType(), True),
    types.StructField("DOLocationID", types.LongType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [5]:
# 3. 데이터 처리 루프
taxi_configs = [('green', green_schema), ('yellow', yellow_schema)]
years = [2020, 2021]

for color, schema in taxi_configs:
    for year in years:
        for month in range(1, 13):
            # 압축 해제된 .parquet 파일들을 찾기 위해 와일드카드 사용
            input_path = f'data/raw/{color}/{year}/{month:02d}/*.parquet'
            output_path = f'data/pq/{color}/{year}/{month:02d}/'

            # 폴더가 있는지 확인
            if not os.path.exists(f'data/raw/{color}/{year}/{month:02d}/'):
                continue

            print(f"작업 중: {color} {year}/{month}")

            try:
                # 순수 Parquet 파일을 읽음
                df = spark.read.schema(schema).parquet(input_path)
                
                # 저장
                df.repartition(4).write.mode('overwrite').parquet(output_path)
            except Exception as e:
                print(f"❌ {year}/{month} 에러 발생: {e}")

print("🎉 모든 작업이 완료되었습니다!")

작업 중: green 2020/1
작업 중: green 2020/2
작업 중: green 2020/3
작업 중: green 2020/4
작업 중: green 2020/5
작업 중: green 2020/6
작업 중: green 2020/7
작업 중: green 2020/8
작업 중: green 2020/9
작업 중: green 2020/10
작업 중: green 2020/11
작업 중: green 2020/12
작업 중: green 2021/1
작업 중: green 2021/2
작업 중: green 2021/3
작업 중: green 2021/4
작업 중: green 2021/5
작업 중: green 2021/6
작업 중: green 2021/7
작업 중: green 2021/8
작업 중: green 2021/9
작업 중: green 2021/10
작업 중: green 2021/11
작업 중: green 2021/12
작업 중: yellow 2020/1
작업 중: yellow 2020/2
작업 중: yellow 2020/3
작업 중: yellow 2020/4
작업 중: yellow 2020/5
작업 중: yellow 2020/6
작업 중: yellow 2020/7
작업 중: yellow 2020/8
작업 중: yellow 2020/9
작업 중: yellow 2020/10
작업 중: yellow 2020/11
작업 중: yellow 2020/12
작업 중: yellow 2021/1
작업 중: yellow 2021/2
작업 중: yellow 2021/3
작업 중: yellow 2021/4
작업 중: yellow 2021/5
작업 중: yellow 2021/6
작업 중: yellow 2021/7
작업 중: yellow 2021/8
작업 중: yellow 2021/9
작업 중: yellow 2021/10
작업 중: yellow 2021/11
작업 중: yellow 2021/12
🎉 모든 작업이 완료되었습니다!
